# M05D: Capstone #2 – Production Support Bot

Build a production support chatbot with budget controls, context management, and response caching.

**Topics:**
- Budget controls + cost monitoring
- Sliding window context management
- Response caching for duplicate messages

---

## 🔧 Step 1: Setup

In [ ]:
import os
import hashlib
from pathlib import Path
from dotenv import load_dotenv

import openai
import tiktoken

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"
    

print(f"✅ Setup complete: Using {MODEL}!")

---

## 🏗️ System Architecture

This bot combines components from M05A, M05B, and M05C into one pipeline.

**Request Flow:** User → budget check → generate response → track cost → log

---

## 📊 Component 1: Token & Cost Tracking

Accurate token counts and budget alerts (from M05A).

In [ ]:
class TokenCounter:
    """Count tokens for text."""
    
    def __init__(self, model=MODEL):
        try:
            self.tokenizer = tiktoken.encoding_for_model(model)
        except KeyError:
            try:
                self.tokenizer = tiktoken.get_encoding("o200k_base")
            except Exception:
                self.tokenizer = tiktoken.get_encoding("cl100k_base")
    
    def count(self, text):
        """Count tokens in text."""
        return len(self.tokenizer.encode(text))


PRICING = {
    "gpt-5": {"input": 1.25, "output": 10.00},
    "gpt-5-mini": {"input": 0.25, "output": 2.00},
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60}
}


class CostTracker:
    """Track cumulative API costs with budget limits."""
    
    def __init__(self, model=MODEL, budget_limit=None):
        self.model = model
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.token_counter = TokenCounter(model)
        self.budget_limit = budget_limit
        self.request_count = 0
    
    def add_turn(self, user_message, assistant_message):
        """Record a turn and return its cost."""
        input_tokens = self.token_counter.count(user_message)
        output_tokens = self.token_counter.count(assistant_message)
        self.total_input_tokens += input_tokens
        self.total_output_tokens += output_tokens
        self.request_count += 1
        return self._calculate_cost(input_tokens, output_tokens)
    
    def _calculate_cost(self, input_tokens, output_tokens):
        """Calculate cost for given token counts."""
        pricing = PRICING.get(self.model, PRICING["gpt-5-mini"])
        input_cost = (input_tokens / 1_000_000) * pricing["input"]
        output_cost = (output_tokens / 1_000_000) * pricing["output"]
        return input_cost + output_cost
    
    def get_total(self):
        """Get total cost so far."""
        return self._calculate_cost(self.total_input_tokens, self.total_output_tokens)
    
    def check_budget(self):
        """Check if within budget."""
        if self.budget_limit is None:
            return True
        return self.get_total() < self.budget_limit
    
    def budget_status(self):
        """Get budget status."""
        current_cost = self.get_total()
        usage_pct = (current_cost / self.budget_limit * 100) if self.budget_limit else 0
        return {
            "current_cost": current_cost,
            "budget_limit": self.budget_limit,
            "usage_percent": usage_pct,
            "tracked_events": self.request_count
        }


# --------------------------------------------------------------
print("✅ Tracking components ready")

---

## 🪟 Component 2: Context Management

Sliding window conversation log (from M05B).

In [ ]:
class ContextWindow:
    """Manage context with sliding window."""
    
    def __init__(self, max_turns=5):
        self.max_turns = max_turns
        self.message_history = []
    
    def add_message(self, speaker, text):
        """Add message and maintain window."""
        # speaker: 'user' or 'assistant'
        self.message_history.append({'speaker': speaker, 'text': text})
        max_messages = self.max_turns * 2
        if len(self.message_history) > max_messages:
            self.message_history[:] = self.message_history[-max_messages:]
    
    def get_messages(self):
        """Get current context."""
        return self.message_history.copy()
    
    def build_context(self):
        """Build context string for API calls."""
        context = ""
        for msg in self.message_history:
            context += f"{msg['speaker']}: {msg['text']}\n"
        return context


# --------------------------------------------------------------
print("✅ Context manager ready")

---

## 🤖 Component 3: Production Support Bot

Orchestrates all components into one pipeline.

In [ ]:
class ProductionSupportBot:
    """Production-ready customer support chatbot."""
    
    INSTRUCTIONS = "You are a customer support agent. Be concise."
    
    def __init__(self, client, max_turns=5, budget_limit=None, model=MODEL):
        self.client = client
        self.model = model
        self.cost_tracker = CostTracker(model, budget_limit)
        self.context = ContextWindow(max_turns)
        self.conversation_log = []
    
    def _generate_response(self, user_message):
        """Generate support response using local context."""
        response = self.client.responses.create(
            model=self.model,
            input=self.context.build_context(),
            instructions=self.INSTRUCTIONS
        )
        return response.output_text.strip()
    
    def chat(self, user_message):
        """Full pipeline: budget check → respond → track cost."""
        if not self.cost_tracker.check_budget():
            return {"response": "Budget exceeded."}
        
        self.context.add_message("user", user_message)
        assistant_message = self._generate_response(user_message)
        self.context.add_message("assistant", assistant_message)
        self.cost_tracker.add_turn(user_message, assistant_message)
        
        result = {"response": assistant_message}
        self.conversation_log.append(result)
        return result
    
    def stats(self):
        """Get session metrics."""
        return {
            "total_turns": len(self.conversation_log),
            "cost": self.cost_tracker.get_total(),
            "budget_status": self.cost_tracker.budget_status()
        }


# --------------------------------------------------------------
print("✅ ProductionSupportBot ready")

---

### Demo: Support Bot in Action

In [ ]:
print("🤖 PRODUCTION SUPPORT BOT DEMO")
print("="*60)

bot = ProductionSupportBot(client, max_turns=5, budget_limit=0.50)

messages = [
    "I was charged twice for my subscription last month!",
    "My order #12345 hasn't arrived and it's been 2 weeks.",
    "How do I reset my password?"
]

for turn_number, message in enumerate(messages, 1):
    print(f"\nTurn {turn_number}: {message}")
    result = bot.chat(message)
    print(f"Response: {truncate_response(result['response'], 400)}")

print("\n" + "="*60)
print("SESSION STATS")
print("="*60)
final = bot.stats()
print(f"Total turns: {final['total_turns']}")
print(f"Total cost: ${final['cost']:.6f}")
print(f"Budget: {final['budget_status']['usage_percent']:.1f}% used")
print("="*60)

---

### 💪 Your Turn: Add Response Caching

The `ResponseCache` class from M05C is provided below. Use it to extend the bot so duplicate messages skip the API call.

- Check cache before calling `_generate_response`
- Store new responses in cache after generating
- Add a `"cached"` flag to the result
- Include cache stats in `stats()`

In [ ]:
# --------------------------------------------------------------
# 💾 ResponseCache (from M05C)
# --------------------------------------------------------------

class ResponseCache:
    """Cache API responses to avoid duplicates."""
    
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def _make_key(self, prompt, instructions, model):
        """Create cache key from prompt + instructions + model."""
        text = f"{model}|{prompt.strip()}|{instructions.strip()}"
        return hashlib.md5(text.encode()).hexdigest()
    
    def get(self, prompt, instructions, model=MODEL):
        """Get cached response if exists."""
        key = self._make_key(prompt, instructions, model)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        return None
    
    def set(self, prompt, instructions, response, model=MODEL):
        """Store response in cache."""
        key = self._make_key(prompt, instructions, model)
        self.cache[key] = response
    
    def stats(self):
        """Get cache statistics."""
        total = self.hits + self.misses
        hit_rate = (self.hits / total * 100) if total else 0
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": hit_rate,
            "size": len(self.cache)
        }


# --------------------------------------------------------------
# 💪 Capstone Exercise: Add Response Caching
# --------------------------------------------------------------
# Objective: Extend ProductionSupportBot to cache duplicate messages.

class CachedSupportBot(ProductionSupportBot):
    """Support bot with response caching."""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # TODO: Initialize a ResponseCache

    def chat(self, user_message):
        """Override to add caching."""
        # TODO: Check budget (same as parent)
        # TODO: Add user message to context
        # TODO: Check cache with self.cache.get(user_message, self.INSTRUCTIONS, self.model)
        # TODO: If cache hit, use cached response
        # TODO: If cache miss, call _generate_response and store in cache
        # TODO: Add assistant message to context
        # TODO: Track cost
        # TODO: Return result dict with "response" and "cached" keys
        pass
    
    def stats(self):
        """Add cache metrics."""
        base_stats = super().stats()
        # TODO: Add self.cache.stats() to base_stats under "cache" key
        return base_stats


# --- Test your implementation ---

# bot = CachedSupportBot(client, max_turns=5, budget_limit=0.50)
# messages = [
#     "I was charged twice for my subscription last month!",
#     "How do I reset my password?",
#     "I was charged twice for my subscription last month!",  # duplicate
#     "How do I reset my password?"  # duplicate
# ]
# for msg in messages:
#     result = bot.chat(msg)
#     source = "CACHE" if result["cached"] else "API"
#     print(f"[{source:5}] {msg[:50]} → {result['response'][:40]}...")
# print(f"\nCache stats: {bot.stats()['cache']}")

print("💡 Implement CachedSupportBot!")
print("💡 See M05D_Solutions.ipynb for complete implementation.")

---

## 🎯 Key Takeaways

**💰 Budget Enforcement Prevents Runaway Costs:**
- Check budget before every request
- Track input and output tokens separately — output costs more

**🪟 Sliding Window Keeps Context Bounded:**
- Fixed memory footprint no matter how long the conversation
- `build_context()` converts the window to an API-ready string

**🏗️ Independent Components, One Orchestrator:**
- Each component is testable on its own
- ProductionSupportBot coordinates the full pipeline
- Each request: budget check → generate response → track cost

---

### 📍 Next Step

**M06A: Function Calling Basics** — Teach the model to call external functions and act on real-world data.

---

## 🔧 Troubleshooting

**Budget hits too fast?**
- Confirm budget is in dollars (not cents)
- Output tokens cost more than input tokens
- Check `budget_status()` to see exact usage

**Cache not hitting?**
- Messages must be EXACTLY identical (including whitespace)
- Instructions must match too
- Check `cache.stats()` to verify hits vs misses

**Context window not trimming?**
- Verify `max_turns` is set correctly
- Check `len(context.get_messages())` after several turns

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---